# 21 · LABEL PIPELINE V1 — disease-aware `-1` + blank-rate routed distillation

Triển khai đúng `LABEL_PIPELINE_V1_DETAILED_PLAN.md`. **Chỉ đổi supervision.** Không đổi RAD-DINO,
patch-only ML-Decoder, resolution, augmentation, subject split, view pooling, EHR, sampler,
checkpoint rule, evaluation labels.

| Stage | Cell | Compute | Nội dung |
|---|---|---|---|
| 0 | C1–C4 | CPU | raw state · blank rate (TRAIN-only) · subject 5-fold · teacher mask |
| 1 | C5–C7 | GPU | 5 teacher fold · OOF probability trên TRAIN |
| 2 | C8–C9 | CPU | final label manifest · `Y_HARD/M_HARD/Y_SOFT/M_SOFT` · diagnostics |
| 3 | C10 | GPU | student loss `ASL(hard) + 0.5·BCE(soft)` — cắm vào vòng train R0 |

**Đọc `REVIEW_LABEL_PIPELINE_V1.md` trước khi chạy C5 trở đi.** Stage 0 (C1–C4) là CPU thuần,
chạy được ngay và in ra bảng routing quyết định có nên đốt GPU cho Stage 1 hay không.

Artifacts → `planB_out/label_pipeline_v1/`. Không ghi đè bất kỳ file nào của nb16/nb17/nb20.

## C0 · Config khoá (plan §18)

In [ ]:
import os, re, json, math, time, copy, pickle, random, shutil, tarfile, hashlib
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
LOCAL_IMG = os.environ.get('LOCAL_IMG', '/content/cxr_local')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
BB_CKPT   = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')            # 04b LP-FT
H5_CKPT   = os.path.join(DATA_DIR, 'token_fusion', 'checkpoints', 'H5_s42_best.pt')   # 06b head
LP1_ROOT  = os.path.join(DATA_DIR, 'label_pipeline_v1')                                # THU MUC MOI

PROJECT, CXR = 'mimic-ehr-v310', 'physionet-data.mimic_cxr_jpg'
RAW_PQ = os.path.join(OUT_DIR, 'cohort_raw_labels.parquet')       # cache cua nb16, CHI DOC

# ================= KHOA: chinh sach nhan (plan §2 · §3 · §18) =================
HARD_U_ONES   = {'Atelectasis', 'Edema'}
SOFT_U_LABELS = {'Cardiomegaly', 'Consolidation', 'Enlarged_Cardiomediastinum', 'Fracture',
                 'Lung_Lesion', 'Lung_Opacity', 'Pleural_Effusion', 'Pleural_Other',
                 'Pneumonia', 'Pneumothorax'}
NO_FINDING    = 'No_Finding'
LAMBDA_SOFT   = 0.5           # plan §10
P_CLIP        = 1e-4          # plan §7: clip(p, 1e-4, 1-1e-4)

# ---- ROUTER CHO blank: theo TY LE '0 THAT' trong so mention (thay cho blank_rate cua plan §3) ----
# Ly do doi: blank_rate cua ca 12 nhan pathology deu > 0.616 nen nguong 0.30 cua plan khong phan
# loai gi — 12/12 deu di TEACHER_SOFT, ke ca nhung nhan ma mention gan nhu toan positive
# (Pleural_Other 94% pos tren 0.94% coverage). Router moi hoi dung cau hoi can hoi: kenh nhan cua
# benh nay CO du negative that de teacher hoc khong?
#   neg_share = n(raw==0) / n(mention),  mention = n(+1) + n(0) + n(-1)
#   neg_share <= 0.30 -> blank = hard 0     (teacher khong du negative -> khong duoc phep ngoai suy)
#   neg_share >  0.30 -> blank = teacher soft
#   No_Finding                -> hard 0 (giu nguyen plan §3)
NEG_SHARE_THRESH = 0.30
BLANK_THRESH_PLAN = 0.30      # nguong blank_rate cua plan §3 — chi con dung o che do 'plan_blank_rate'

# ---- CONG TAC ARM: doi DUY NHAT hang so nay de chuyen giua cac huong so sanh ----
#   'neg_share'        A3 · neg_share > 0.30 -> teacher soft, con lai hard 0   (huong 3)
#   'all_soft'         A1 · MOI nhan pathology blank -> teacher soft
#                           (= plan goc §3, vi blank_rate cua ca 12 nhan deu > 0.616)
#   'all_hard_zero'    A2 · MOI blank -> hard 0 (= H0 historical, chinh la nhan cua R0/nb16)
#   'all_one'          A0 · MOI blank -> hard 1 (degenerate, chi de doi chung)
#   'plan_blank_rate'  router blank_rate <= 0.30 cua plan, giu de tai lap
# -1 GIU NGUYEN chien luoc disease-aware cua plan o MOI arm.
BLANK_ROUTE_MODE = 'all_hard_zero'
assert BLANK_ROUTE_MODE in ('neg_share', 'all_soft', 'all_hard_zero', 'all_one', 'plan_blank_rate')
BLANK_ROUTE_RULE = {'neg_share': f'explicit_negative_share > {NEG_SHARE_THRESH}',
                    'all_soft': 'all pathology blank -> TEACHER_SOFT',
                    'all_hard_zero': 'all blank -> HARD_ZERO (H0 historical)',
                    'all_one': 'all pathology blank -> HARD_ONE (degenerate)',
                    'plan_blank_rate': f'blank_rate > {BLANK_THRESH_PLAN}'}[BLANK_ROUTE_MODE]
ARM_TAG = dict(neg_share='A3_negshare', all_soft='A1_allsoft', all_hard_zero='A2_allzero',
               all_one='A0_allone', plan_blank_rate='A1_planrate')[BLANK_ROUTE_MODE]

# ---- TEACHER tach khoi ARM de cache dung chung ----
#   'shared_blank0'  teacher hoc voi blank -> hard 0 cho MOI nhan (coverage 96-100% moi kenh,
#                    dung phan phoi R0). Dung chung cho A0/A2/A3 -> chi train 1 lan.
#   'arm_masked'     teacher mask blank cua cac nhan TEACHER_SOFT, dung chu plan §5. Rieng moi arm.
#   'auto'           A1_allsoft/A1_planrate -> arm_masked ; con lai -> shared_blank0
# LUU Y (xem REVIEW §4-bis): voi A3, teacher shared_blank0 da duoc day blank=0 nen soft target
# tren blank se lech ve 0. No van phat hien duoc omission (model bat dong voi nhan), nhung
# yeu hon teacher masked. Doi TEACHER_MODE='arm_masked' neu muon A3 khong mang prior do.
TEACHER_MODE = 'auto'
if TEACHER_MODE == 'auto':
    TEACHER_MODE = ('arm_masked' if BLANK_ROUTE_MODE in ('all_soft', 'plan_blank_rate')
                    else 'shared_blank0')
assert TEACHER_MODE in ('shared_blank0', 'arm_masked')
TEACHER_TAG = 'T_blank0' if TEACHER_MODE == 'shared_blank0' else f'T_masked_{ARM_TAG}'

# ================= KHOA: teacher (plan §6) =================
TEACHER_SEED  = 42
K_FOLD        = 5
TEACHER_EPOCHS = 8            # = EPOCHS cua R0; KHONG chon checkpoint bang fold held-out (xem review §3)
BATCH = int(os.environ.get('LP1_BATCH', 64))   # o C0 de C7-C10 dung duoc ma khong can chay C6
TEACHER_CKPT_RULE = 'fixed_last_epoch'

# ================= KHOA: R0 (verbatim nb16 — KHONG duoc doi) =================
EPOCHS, EFF = 8, 256
LR_BLOCKS, LR_HEAD = 1e-5, 1e-5
CLIP_BLOCKS, CLIP_HEAD = 5.0, 1.0
WD = 1e-2
WARMUP_FRAC = 0.05
HEAD_WARMUP_EPOCHS = 1
ASL_GP, ASL_GN, ASL_CLIP, ASL_EPS = 1, 4, 0.05, 1e-8
W_F, W_L   = 0.7, 0.3          # view pooling khoa
MACRO_TOL  = 0.0015
N_BOOT     = 2000
RARE3 = ['Pleural_Other', 'Fracture', 'Lung_Lesion']
WORST3_FIXED = ['Fracture', 'Pleural_Other', 'Enlarged_Cardiomediastinum']
STUDENT_SEEDS = [42, 1337, 2024]

ACCUM = max(1, EFF // BATCH)

PREREG = dict(
    plan='LABEL_PIPELINE_V1_DETAILED_PLAN.md',
    review='REVIEW_LABEL_PIPELINE_V1.md',
    scope='supervision-only; kien truc/split/preprocessing/view-pooling/schedule = R0 nb16',
    hard_u_ones=sorted(HARD_U_ONES), soft_u_labels=sorted(SOFT_U_LABELS),
    arm=ARM_TAG, blank_route_mode=BLANK_ROUTE_MODE, blank_route_rule=BLANK_ROUTE_RULE,
    neg_share_threshold=NEG_SHARE_THRESH, blank_threshold_plan=BLANK_THRESH_PLAN,
    blank_rate_source='TRAIN raw labels only (cohort da select)',
    lambda_soft=LAMBDA_SOFT, teacher_seed=TEACHER_SEED, k_fold=K_FOLD,
    teacher_mode=TEACHER_MODE, teacher_tag=TEACHER_TAG,
    teacher_ckpt_rule=TEACHER_CKPT_RULE,
    teacher_loss='sum(M_T * ASL) / (B_actual * 13)',
    student_loss='[sum(M_hard*ASL) + 0.5*sum(M_soft*BCE)] / (B_actual * 13)',
    eval='historical VAL/TEST labels (+1/-1 -> pos, 0/blank -> neg); primary Macro mAP',
    gate='Macro >= +0.002 OR Rare3 >= +0.002; NonTarget10 >= -0.0015; >=2/3 seed cung chieu')
LP1     = os.path.join(LP1_ROOT, ARM_TAG)              # artifacts rieng tung arm
TEA_DIR = os.path.join(LP1_ROOT, '_teachers', TEACHER_TAG)   # teacher dung chung, cache theo TAG
for _d in (LP1, f'{LP1}/metrics', TEA_DIR, f'{TEA_DIR}/checkpoints', f'{TEA_DIR}/preds'):
    os.makedirs(_d, exist_ok=True)
json.dump(PREREG, open(f'{LP1}/preregistration.json', 'w'), indent=1)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '|', torch.cuda.get_device_name(0) if dev == 'cuda' else 'CPU')
if dev == 'cuda': print(f'  VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
print(f'ARM {ARM_TAG} | blank router: {BLANK_ROUTE_RULE}')
print(f'teacher {TEACHER_TAG} ({TEACHER_MODE}) | {K_FOLD} fold seed {TEACHER_SEED}')
print(f'lambda_soft {LAMBDA_SOFT} | arm -> {LP1} | teacher -> {TEA_DIR}')

## C1 · Cohort + **nhãn thô bốn trạng thái** *(Stage 0)*

Đọc `cohort_study_level.parquet` (nhãn historical U-Ones) và `cohort_raw_labels.parquet`
(cache CheXpert thô của nb16). Assert U-Ones(raw) tái tạo đúng nhãn cohort — nếu fail thì
mọi thứ phía sau vô nghĩa.

State code: `1` = `+1` · `0` = explicit negative · `-1` = uncertain · `2` = blank.

In [ ]:
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; C = len(LABELS); assert C == 13
assert set(LABELS) == HARD_U_ONES | SOFT_U_LABELS | {NO_FINDING}, \
    f'nhan cohort khong khop chinh sach: {sorted(set(LABELS) ^ (HARD_U_ONES|SOFT_U_LABELS|{NO_FINDING}))}'

ehr = (pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet'),
                       columns=['study_id', 'subject_id', 'split', 'setting'] + LABELS)
       .sort_values('study_id').reset_index(drop=True))
N = len(ehr)
Y_STUDY   = ehr[LABELS].to_numpy('float32')                 # historical U-Ones (H0)
STUDY_ROW = {int(s): i for i, s in enumerate(ehr.study_id.to_numpy())}
SPLIT     = ehr.split.to_numpy()
SUBJ      = ehr.subject_id.to_numpy()
IS_TRAIN  = SPLIT == 'train'
N_TR      = int(IS_TRAIN.sum())
print(f'cohort {N:,} study | {dict(zip(*np.unique(SPLIT, return_counts=True)))}')

cf = pickle.load(open(os.path.join(OUT_DIR, 'class_freq.pkl'), 'rb'))
N_POS_TRAIN = np.asarray(cf['class_freq'], 'float64'); N_TRAIN_CF = int(cf['num_train'])
assert N_TRAIN_CF == N_TR, f'class_freq.num_train {N_TRAIN_CF} != {N_TR}'
TIER = np.where(N_POS_TRAIN/N_TR < 0.02, 'tail', np.where(N_POS_TRAIN/N_TR < 0.10, 'medium', 'head'))
RARE3_IDX  = np.array([LABELS.index(l) for l in RARE3])
NONTGT_IDX = np.setdiff1d(np.arange(C), RARE3_IDX)
TAIL_IDX   = np.where(TIER == 'tail')[0]
W3_IDX     = np.array([LABELS.index(l) for l in WORST3_FIXED])

# ---------------- nhan tho ----------------
if os.path.exists(RAW_PQ):
    rawdf = pd.read_parquet(RAW_PQ); print(f'raw labels <- cache {RAW_PQ}')
else:
    from google.cloud import bigquery
    try:
        from google.colab import auth; auth.authenticate_user()
    except Exception:
        pass
    print('query BigQuery ...')
    chex = bigquery.Client(project=PROJECT, location='US').query(
        f'SELECT * FROM `{CXR}.chexpert`').to_dataframe()
    _nm = lambda s: re.sub(r'[^a-z]', '', str(s).lower())
    _col = {_nm(c): c for c in chex.columns}
    assert not [l for l in LABELS if _nm(l) not in _col], 'khong map duoc cot chexpert'
    rawdf = chex.drop_duplicates('study_id').set_index('study_id')[[_col[_nm(l)] for l in LABELS]]
    rawdf.columns = [f'raw_{l}' for l in LABELS]
    rawdf = rawdf.reset_index(); rawdf.to_parquet(RAW_PQ, index=False)
    print(f'raw labels -> {RAW_PQ} ({len(rawdf):,} rows) [FILE MOI]')

RAW = (rawdf.drop_duplicates('study_id').set_index('study_id')
       .reindex(ehr.study_id.to_numpy())[[f'raw_{l}' for l in LABELS]].to_numpy('float64'))
assert RAW.shape == (N, C)
_y = np.where(np.isnan(RAW), 0.0, np.where(RAW == -1, 1.0, RAW)).astype('float32')
_bad = int((_y != Y_STUDY).sum())
print(f'U-Ones(raw) == cohort labels ? {_bad == 0}  ({_bad} o lech)')
assert _bad == 0, 'nhan tho KHONG tai tao duoc nhan cohort -> dung chay tiep'

STATE = np.where(np.isnan(RAW), 2, RAW).astype('int8')       # 1 / 0 / -1 / 2(blank)
IS_BLANK = STATE == 2
print(f'trang thai (toan cohort): +1 {int((STATE==1).sum()):>9,} | 0 {int((STATE==0).sum()):>9,} '
      f'| -1 {int((STATE==-1).sum()):>9,} | blank {int(IS_BLANK.sum()):>9,}')

# ---------------- view manifest (giong nb16) ----------------
vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal', 'lateral'])]
_dup = [c for c in ('subject_id', 'split', 'setting') if c in vm.columns]
MAN_ALL = (vm.drop(columns=_dup).merge(ehr[['study_id', 'subject_id', 'split']], on='study_id',
                                       how='inner').sort_values('dicom_id').reset_index(drop=True))
print(f'anh (view_map day du): {len(MAN_ALL):,} | {dict(MAN_ALL.split.value_counts())}')
print('\nNOTE: VAL/TEST khong duoc dung o BAT KY buoc nao cua Stage 0/1/2.')

## C2 · **Stage 0a** — blank rate trên TRAIN, khoá routing một lần *(plan §3)*

\[ r^{blank}_c = N_{TRAIN}(raw=blank)/N_{TRAIN}(study) \]

**Router đã đổi so với plan §3** (xem `REVIEW_LABEL_PIPELINE_V1.md` §2/§4): điều hướng theo
`neg_share = n(raw==0) / n(mention)` chứ không theo `blank_rate`.

```
neg_share <= 0.30 -> blank = hard 0        (kênh nhãn không đủ negative thật -> cấm ngoại suy)
neg_share >  0.30 -> blank = teacher soft
No_Finding        -> hard 0                (giữ nguyên plan §3)
```

Lý do: `blank_rate` của cả 12 nhãn pathology đều `> 0.616` nên ngưỡng 0.30 của plan không phân loại
gì — 12/12 đều đi `TEACHER_SOFT`, kể cả nhãn mà mention gần như toàn positive. `blank_rate` vẫn được
tính và lưu đầy đủ (§8 yêu cầu), chỉ không còn điều hướng. Không tính lại theo fold/seed/VAL/TEST.

Cell này cũng in **cổng quyết định trước GPU**: tỉ lệ cell teacher-supervised, prevalence positive
trong tập hard của teacher, và số cell sẽ trở thành soft target. Đọc kỹ trước khi chạy C7.

In [ ]:
ST_TR = STATE[IS_TRAIN]                                   # [N_TR, C] — CHI TRAIN
n_pos   = (ST_TR ==  1).sum(0); n_neg0  = (ST_TR == 0).sum(0)
n_unc   = (ST_TR == -1).sum(0); n_blank = (ST_TR == 2).sum(0)
assert (n_pos + n_neg0 + n_unc + n_blank == N_TR).all()
blank_rate = n_blank / N_TR

n_ment    = n_pos + n_neg0 + n_unc
neg_share = n_neg0 / np.maximum(n_ment, 1)                # ty le '0 that' trong so mention
if BLANK_ROUTE_MODE == 'neg_share':
    policy = np.where(neg_share > NEG_SHARE_THRESH, 'TEACHER_SOFT', 'HARD_ZERO')
elif BLANK_ROUTE_MODE == 'plan_blank_rate':
    policy = np.where(blank_rate <= BLANK_THRESH_PLAN, 'HARD_ZERO', 'TEACHER_SOFT')
elif BLANK_ROUTE_MODE == 'all_soft':
    policy = np.full(C, 'TEACHER_SOFT')
elif BLANK_ROUTE_MODE == 'all_hard_zero':
    policy = np.full(C, 'HARD_ZERO')
elif BLANK_ROUTE_MODE == 'all_one':
    policy = np.full(C, 'HARD_ONE')
policy = np.asarray(policy, dtype=object)
policy[LABELS.index(NO_FINDING)] = 'HARD_ZERO'            # plan §3: No Finding blank -> hard 0
_policy_plan = np.where(blank_rate <= BLANK_THRESH_PLAN, 'HARD_ZERO', 'TEACHER_SOFT')
_policy_plan[LABELS.index(NO_FINDING)] = 'HARD_ZERO'      # router cua plan, de doi chieu
NEED_TEACHER = bool((policy == 'TEACHER_SOFT').any() or len(SOFT_U_LABELS))
print(f'can teacher OOF ? {NEED_TEACHER}  (soft tu blank: {int((policy=="TEACHER_SOFT").sum())} nhan'
      f' | soft tu -1: {len(SOFT_U_LABELS)} nhan)')

RAWSUM = pd.DataFrame(dict(
    label=LABELS, tier=TIER, n_train_studies=N_TR,
    n_pos=n_pos, n_neg0=n_neg0, n_unc=n_unc, n_blank=n_blank,
    n_mention=n_ment, neg_share=neg_share,
    blank_rate=blank_rate, blank_policy=policy, blank_policy_plan_rule=_policy_plan,
    unc_policy=[('HARD_ONE' if l in HARD_U_ONES else 'MASK' if l == NO_FINDING else 'TEACHER_SOFT')
                for l in LABELS],
    hist_prevalence=(n_pos + n_unc) / N_TR))
RAWSUM.to_csv(f'{LP1}/raw_train_label_summary.csv', index=False)
json.dump(dict(rule=BLANK_ROUTE_RULE, neg_share_threshold=NEG_SHARE_THRESH,
               blank_rate_threshold_plan_superseded=BLANK_THRESH_PLAN,
               by_label={l: dict(n_train_studies=int(N_TR), n_blank=int(b), blank_rate=float(r),
                                 n_mention=int(m), n_neg0=int(g), neg_share=float(q),
                                 blank_policy=str(p))
                         for l, b, r, m, g, q, p in zip(LABELS, n_blank, blank_rate, n_ment,
                                                        n_neg0, neg_share, policy)}),
          open(f'{LP1}/blank_policy_by_label.json', 'w'), indent=1)

print('=== RAW TRAIN LABEL SUMMARY (plan §15) ===')
print(RAWSUM.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

BLANK_SOFT = np.array([p == 'TEACHER_SOFT' for p in policy])
print('\nARM ' + ARM_TAG + ' | ' + ' | '.join(
    f'{k} {int((policy == k).sum())}/{C}' for k in ('HARD_ZERO', 'TEACHER_SOFT', 'HARD_ONE')))

# ---------------- cong quyet dinh truoc GPU ----------------
hard_cells = np.where(policy == 'TEACHER_SOFT', 0, n_blank) + n_pos + n_neg0 \
             + np.array([n_unc[j] if LABELS[j] in HARD_U_ONES else 0 for j in range(C)])
hard_pos   = n_pos + np.array([n_unc[j] if LABELS[j] in HARD_U_ONES else 0 for j in range(C)]) \
             + np.where(policy == 'HARD_ONE', n_blank, 0)
soft_cells = np.where(policy == 'TEACHER_SOFT', n_blank, 0) \
             + np.array([n_unc[j] if LABELS[j] in SOFT_U_LABELS else 0 for j in range(C)])
GATE = pd.DataFrame(dict(label=LABELS, neg_share=neg_share, blank_rate=blank_rate,
                         blank_policy=policy,
                         teacher_hard_cells=hard_cells, teacher_hard_cov=hard_cells/N_TR,
                         teacher_hard_pos_rate=hard_pos/np.maximum(hard_cells, 1),
                         soft_cells=soft_cells, soft_frac=soft_cells/N_TR,
                         hist_prevalence=(n_pos+n_unc)/N_TR))
GATE.to_csv(f'{LP1}/metrics/stage0_gate.csv', index=False)
print('\n=== CONG QUYET DINH TRUOC GPU ===')
print(GATE.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print(f'\ntong soft cell / tong cell TRAIN: {int(soft_cells.sum()):,} / {N_TR*C:,} '
      f'= {soft_cells.sum()/(N_TR*C):.1%}')
# ---------------- thanh phan 1-that / 0-that trong phan nhan CO TON TAI ----------------
# Cau hoi: voi nhan hiem, phan duoc nhac toi la 1 that hay 0 that? Ty le bao nhieu?
# prev_MAR = prevalence NEU blank la missing-at-random (blank co cung ty le 1/0 nhu mention).
n_ment = n_pos + n_neg0 + n_unc
MENT = pd.DataFrame(dict(label=LABELS, n_mention=n_ment, frac_cohort=n_ment/N_TR,
                         pct_pos=n_pos/np.maximum(n_ment, 1), pct_neg0=n_neg0/np.maximum(n_ment, 1),
                         pct_unc=n_unc/np.maximum(n_ment, 1),
                         pos_per_neg=(n_pos+n_unc)/np.maximum(n_neg0, 1),
                         prev_hist=(n_pos+n_unc)/N_TR,
                         prev_if_MAR=(n_pos+n_unc)/np.maximum(n_ment, 1)))
MENT['blowup'] = MENT.prev_if_MAR / MENT.prev_hist.clip(lower=1e-9)
MENT.to_csv(f'{LP1}/metrics/mention_composition.csv', index=False)
print('\n=== THANH PHAN NHAN CO TON TAI (1 that vs 0 that) — TRAIN cohort ===')
print(MENT.sort_values('n_mention').to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print('  prev_if_MAR = prevalence NEU blank co cung ty le 1/0 nhu phan duoc nhac toi.')
print('  blowup >> 1 => gia thiet missing-at-random SAI => blank ~ 0 that (reporting bias:')
print('  bac si chi ghi dau hieu hiem khi CO). Xem REVIEW_LABEL_PIPELINE_V1.md §2-bis.')

# ---------------- doi chieu router moi vs router cua plan §3 ----------------
_flip = [(l, a, b) for l, a, b in zip(LABELS, _policy_plan, policy) if a != b]
print(f'\n=== ROUTER: {BLANK_ROUTE_RULE} ===')
for _k in ('TEACHER_SOFT', 'HARD_ZERO', 'HARD_ONE'):
    _ls = sorted(np.asarray(LABELS)[policy == _k].tolist())
    if _ls: print(f'  blank -> {_k:<13}: {_ls}')
print(f'  doi so voi router blank_rate<=0.30 cua plan: {len(_flip)} nhan')
for l, a, b in _flip:
    print(f'    {l:<28} {a} -> {b}')
_marg = GATE[(GATE.neg_share - NEG_SHARE_THRESH).abs() < 0.02]
if len(_marg):
    print(f'\n!! SAT NGUONG (|neg_share - {NEG_SHARE_THRESH}| < 0.02) — route cua cac nhan nay mong manh:')
    for r in _marg.itertuples():
        _n_flip = int(abs(r.neg_share - NEG_SHARE_THRESH) * (n_ment[LABELS.index(r.label)]))
        print(f'    {r.label:<28} neg_share {r.neg_share:.4f} -> {r.blank_policy} '
              f'(chi can ~{_n_flip:,} study doi trang thai la lat route)')

_thin = GATE[(GATE.teacher_hard_cov < 0.05) | (GATE.teacher_hard_pos_rate > 0.90)]
if len(_thin):
    print('\n!! CANH BAO: cac nhan sau co teacher hard supervision qua mong hoac gan nhu toan positive')
    print('   -> ASL se day p ve ~1 tren MOI study, soft target se bom prevalence len nhieu lan.')
    print('   Doc REVIEW_LABEL_PIPELINE_V1.md §2 truoc khi chay C7.')
    print(_thin[['label', 'teacher_hard_cells', 'teacher_hard_cov', 'teacher_hard_pos_rate']]
          .to_string(index=False, float_format=lambda v: f'{v:.4f}'))

## C3 · **Stage 0b** — subject 5-fold, cân bằng multilabel *(plan §6.1)*

Greedy group-assignment: subject nặng nhất trước, gán vào fold làm nhỏ nhất
`max_c (load_c + subject_load_c)/target_c`. Deterministic với `TEACHER_SEED`.
Yêu cầu: không subject overlap · một fold map dùng cho mọi label.

In [ ]:
TR_ROWS = np.where(IS_TRAIN)[0]
SID_TR  = ehr.study_id.to_numpy()[TR_ROWS]          # da sort tang dan
SUBJ_TR = SUBJ[TR_ROWS]

def balanced_subject_folds(subj, load_study, K, seed):
    '''Iterative stratification o muc GROUP=subject (Sechidis 2011, ban group-aware).

    load_study [n_study, C]: so cell co mention (pos/neg/unc) — dai luong can can bang.
    Moi subject duoc gan tron ven vao mot fold; uu tien nhan HIEM nhat ma subject do cham vao,
    tie-break bang fold dang thieu study nhat -> can bang ca nhan lan so study.'''
    su, inv = np.unique(subj, return_inverse=True)
    load = np.zeros((len(su), load_study.shape[1]), 'float64')
    np.add.at(load, inv, load_study.astype('float64'))
    nst = np.bincount(inv, minlength=len(su)).astype('float64')
    rng = np.random.default_rng(seed)
    order = np.lexsort((rng.random(len(su)), -load.sum(1)))        # nang -> nhe, tie-break seeded
    rank = np.empty(load.shape[1], 'int64')
    rank[np.argsort(load.sum(0), kind='stable')] = np.arange(load.shape[1])   # 0 = hiem nhat
    need   = np.tile(load.sum(0)/K, (K, 1))
    need_n = np.full(K, nst.sum()/K)
    f_su = np.empty(len(su), 'int64')
    for i in order:
        cols = np.flatnonzero(load[i] > 0)
        if len(cols):
            c = int(cols[np.argmin(rank[cols])])
            cand = np.flatnonzero(need[:, c] >= need[:, c].max() - 1e-9)
        else:
            cand = np.arange(K)
        if len(cand) > 1:
            cand = cand[need_n[cand] >= need_n[cand].max() - 1e-9]
        k = int(cand[0]); f_su[i] = k; need[k] -= load[i]; need_n[k] -= nst[i]
    return f_su[inv], su, f_su

# plan §6.1: can bang "multilabel positive/explicit-negative counts" -> hai khoi cot rieng.
# Cot hiem nhat (vd Pleural_Other explicit-negative, chi 83 study TRAIN) duoc rai truoc,
# nen gate §15 o C4 co co hoi pass cao nhat.
LOAD = np.hstack([(ST_TR == 1), (ST_TR == 0)]).astype('float64')          # [N_TR, 2C]
FOLD_TR, _su, _fsu = balanced_subject_folds(SUBJ_TR, LOAD, K_FOLD, TEACHER_SEED)

for k in range(K_FOLD):
    a = set(SUBJ_TR[FOLD_TR == k]); b = set(SUBJ_TR[FOLD_TR != k])
    assert a.isdisjoint(b), f'fold {k} chong lap subject'
FOLD_OF_ROW = np.full(N, -1, 'int64'); FOLD_OF_ROW[TR_ROWS] = FOLD_TR

FOLDS = pd.DataFrame(dict(study_id=SID_TR, subject_id=SUBJ_TR, study_row=TR_ROWS, fold=FOLD_TR))
FOLDS.to_parquet(f'{LP1}/train_teacher_folds.parquet', index=False)
print(f'{len(_su):,} subject TRAIN -> {K_FOLD} fold, subject-disjoint OK')
print('  study/fold  :', [int((FOLD_TR == k).sum()) for k in range(K_FOLD)])
print('  subject/fold:', [int((_fsu == k).sum()) for k in range(K_FOLD)])
_bal = pd.DataFrame({f'fold{k}': (ST_TR[FOLD_TR == k] == 1).sum(0) for k in range(K_FOLD)},
                    index=LABELS)
_bal['cv_pos'] = _bal.std(1) / _bal.mean(1).clip(lower=1)
print('\n=== n(+1) theo fold (can bang multilabel) ===')
print(_bal.to_string(float_format=lambda v: f'{v:.4f}'))
_lo = _bal[_bal.cv_pos > 0.15]
if len(_lo):
    print(f'!! {len(_lo)} nhan co CV(n_pos) > 15% giua cac fold: {list(_lo.index)} '
          '(chi canh bao — gate cung la §15 o C4)')
assert (_bal['cv_pos'] < 0.30).all(), 'fold lech positive > 30% -> xem lai balanced_subject_folds'

## C4 · **Stage 0c** — teacher supervision manifest *(plan §5)*

```
raw +1                         -> y=1, m=1
raw  0                         -> y=0, m=1
Atelectasis/Edema raw -1       -> y=1, m=1
blank cua nhan HARD_ZERO       -> y=0, m=1
No_Finding +1 -> 1 | 0/blank -> 0
--- masked (m=0) ---
raw -1 cua SOFT_U_LABELS · raw -1 cua No_Finding · blank cua nhan TEACHER_SOFT
```

Gate `§15`: nếu một cặp *(fold, label)* thiếu hẳn positive **hoặc** negative hard supervision → **STOP**.

In [ ]:
# Teacher co POLICY RIENG, khong nhat thiet trung policy cua arm.
if TEACHER_MODE == 'shared_blank0':
    policy_teacher = np.full(C, 'HARD_ZERO', dtype=object)   # moi blank -> hard 0
else:
    policy_teacher = policy.copy()                            # dung chu plan §5
print(f'teacher policy ({TEACHER_MODE}): ' + ' | '.join(
    f'{k} {int((policy_teacher == k).sum())}' for k in ('HARD_ZERO', 'TEACHER_SOFT', 'HARD_ONE')))

Y_T = np.zeros((N, C), 'float32')          # target teacher (chi TRAIN co nghia)
M_T = np.zeros((N, C), 'float32')          # mask teacher
for j, l in enumerate(LABELS):
    st = STATE[:, j]
    pos, neg, unc, bl = st == 1, st == 0, st == -1, st == 2
    Y_T[pos, j] = 1.0; M_T[pos, j] = 1.0
    Y_T[neg, j] = 0.0; M_T[neg, j] = 1.0
    if l in HARD_U_ONES:
        Y_T[unc, j] = 1.0; M_T[unc, j] = 1.0
    else:
        M_T[unc, j] = 0.0                                   # SOFT_U_LABELS + No_Finding -> mask
    if policy_teacher[j] == 'HARD_ZERO':                    # gom ca No_Finding
        Y_T[bl, j] = 0.0; M_T[bl, j] = 1.0
    elif policy_teacher[j] == 'HARD_ONE':
        Y_T[bl, j] = 1.0; M_T[bl, j] = 1.0
    else:
        M_T[bl, j] = 0.0                                    # TEACHER_SOFT -> mask o teacher
assert ((M_T == 0) | (M_T == 1)).all() and ((Y_T == 0) | (Y_T == 1)).all()

TS = []
for k in range(K_FOLD):
    r = TR_ROWS[FOLD_TR == k]
    for j, l in enumerate(LABELS):
        m = M_T[r, j] == 1
        TS.append(dict(fold=k, label=l, hard_pos=int((Y_T[r, j][m] == 1).sum()),
                       hard_neg=int((Y_T[r, j][m] == 0).sum()), masked=int((~m).sum()),
                       n_study=len(r)))
TSD = pd.DataFrame(TS)
TSD['pos_prevalence'] = TSD.hard_pos / (TSD.hard_pos + TSD.hard_neg).clip(lower=1)
TSD['hard_cov'] = (TSD.hard_pos + TSD.hard_neg) / TSD.n_study
os.makedirs(TEA_DIR, exist_ok=True)          # tu tao lai: song sot ca khi kernel giu C0 cu
TSD.to_parquet(f'{TEA_DIR}/teacher_supervision_manifest.parquet', index=False)
np.save(f'{TEA_DIR}/teacher_Y.npy', Y_T); np.save(f'{TEA_DIR}/teacher_M.npy', M_T)

# fold map phai giong nhau thi teacher moi dung chung duoc -> khoa bang hash
_fold_sha = hashlib.sha256(FOLD_TR.tobytes()).hexdigest()
_fp = f'{TEA_DIR}/fold_map.sha256'
if os.path.exists(_fp):
    assert open(_fp).read().strip() == _fold_sha, (
        f'fold map khac voi teacher da cache o {TEA_DIR} -> KHONG duoc dung chung OOF. '
        'Xoa thu muc teacher hoac doi TEACHER_TAG.')
    print(f'fold map khop teacher da cache ({_fold_sha[:12]}...)')
else:
    open(_fp, 'w').write(_fold_sha)

print('=== TEACHER HARD COVERAGE theo label (gop 5 fold) ===')
_g = TSD.groupby('label', sort=False).agg(hard_pos=('hard_pos', 'sum'), hard_neg=('hard_neg', 'sum'),
                                          masked=('masked', 'sum'))
_g['pos_prevalence'] = _g.hard_pos / (_g.hard_pos + _g.hard_neg).clip(lower=1)
_g['hard_cov'] = (_g.hard_pos + _g.hard_neg) / N_TR
print(_g.loc[LABELS].to_string(float_format=lambda v: f'{v:.4f}'))

_bad = TSD[(TSD.hard_pos == 0) | (TSD.hard_neg == 0)]
print(f'\n=== GATE §15: fold-label thieu positive HOAC negative: {len(_bad)} ===')
if len(_bad):
    print(_bad.to_string(index=False))
assert len(_bad) == 0, 'STOP: sua group-stratified fold assignment (plan §15)'
_thin = TSD[(TSD.hard_pos + TSD.hard_neg) < 200]
if len(_thin):
    print(f'!! {len(_thin)} cap (fold,label) co <200 hard cell -> teacher se rat nhieu (khong phai STOP)')
    print(_thin.sort_values('hard_pos').to_string(index=False))
print('\nteacher_supervision_manifest.parquet OK')

## C5 · Stage ảnh + dataset *(bắt đầu phần GPU — verbatim nb16 C0b/C5)*

Ảnh, preprocessing, transform **không đổi**. Dataset trả `y_T` + `m_T` thay cho `y` U-Ones —
đây là điểm khác duy nhất so với nb16.

In [ ]:
os.makedirs(LOCAL_IMG, exist_ok=True)
_have = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
if _have > 1000 and not os.path.isdir(SHARD_DIR):
    print(f'{_have:,} anh da co san o {LOCAL_IMG}; khong co SHARD_DIR -> bo qua staging')
else:
    tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar')) if os.path.isdir(SHARD_DIR) else []
    assert tars or _have > 1000, f'khong co shard o {SHARD_DIR} va {LOCAL_IMG} rong -> can data'
    todo = [f for f in tars if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_{f}'))]
    print(f'shards: {len(tars)} | to stage: {len(todo)} | da co {_have:,} anh')
    for f in tqdm(todo, desc='staging shards', unit='shard'):
        tmp = f'/content/{f}'
        shutil.copyfile(os.path.join(SHARD_DIR, f), tmp)
        with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
        os.remove(tmp); open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()
    print('local images:', len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')]))

In [ ]:
from transformers import AutoImageProcessor
from torchvision import transforms as T
from PIL import Image
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = proc.crop_size['height']; MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(MODEL_SIZE, scale=(0.85,1.0), ratio=(0.95,1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])
eval_tf  = T.Compose([T.CenterCrop(MODEL_SIZE), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])

_before = len(MAN_ALL)
man = MAN_ALL[MAN_ALL.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))
              ].sort_values('dicom_id').reset_index(drop=True)
assert len(man) and (_before-len(man))/_before < 0.01, f'{_before-len(man)} anh thieu -> stage anh'
man['study_row'] = man.study_id.map(STUDY_ROW).astype('int64')
man['fold'] = FOLD_OF_ROW[man.study_row.to_numpy()]
IDX = {s: np.where(man.split.to_numpy() == s)[0] for s in ('train', 'val', 'test')}
print(f'anh theo split: { {s: int(len(IDX[s])) for s in IDX} }')

# ---- moi TRAIN study PHAI co it nhat 1 anh, neu khong khong the co OOF (plan §7) ----
_cov = np.zeros(N, bool); _cov[man.study_row.to_numpy()] = True
_miss = TR_ROWS[~_cov[TR_ROWS]]
print(f'TRAIN study khong co anh nao: {len(_miss):,}')
assert len(_miss) == 0, ('STOP: khong the sinh OOF cho study khong co anh. Stage them shard '
                         f'hoac xem lai view_map. Vi du study_id: {ehr.study_id.to_numpy()[_miss][:5]}')

class ImgDS(Dataset):
    '''Anh + teacher target/mask + study_row. KHONG co EHR.'''
    def __init__(s, idx, tf, Y, M): s.idx, s.tf, s.Y, s.M = np.asarray(idx), tf, Y, M
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        r = man.iloc[s.idx[k]]
        im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
        im = Image.merge('RGB', (im, im, im))
        tr = int(r.study_row)
        return dict(image=s.tf(im), y=torch.from_numpy(s.Y[tr]), m=torch.from_numpy(s.M[tr]), sr=tr)

def collate(bb):
    return dict(image=torch.stack([x['image'] for x in bb]), y=torch.stack([x['y'] for x in bb]),
                m=torch.stack([x['m'] for x in bb]),
                sr=torch.tensor([x['sr'] for x in bb], dtype=torch.int64))

def loader(idx, tf, shuffle, bs, Y, M, workers=4):
    return DataLoader(ImgDS(idx, tf, Y, M), batch_size=bs, shuffle=shuffle, collate_fn=collate,
                      num_workers=workers, pin_memory=(dev == 'cuda'), persistent_workers=(workers > 0))

to_dev = lambda b: {k: v.to(dev, non_blocking=True) for k, v in b.items()}
_b = to_dev(collate([ImgDS(IDX['train'], eval_tf, Y_T, M_T)[k] for k in range(4)]))
assert (Y_T[_b['sr'].cpu().numpy()] == _b['y'].cpu().numpy()).all()
print('dataset OK | image', tuple(_b['image'].shape), '| y', tuple(_b['y'].shape),
      '| m mean', float(_b['m'].mean()))

## C6 · Model + loss — **verbatim R0** *(blocks 0–8 frozen · 9–11 + backbone LN + ML-Decoder · patch-only)*

`CellWASL(x, y, w)` giống hệt nb16; truyền `w = M` cho ra đúng công thức teacher của plan §5:
\[ L_T=\frac{\sum M^T\,L_{ASL}}{B_{actual}\times 13} \]

In [ ]:
from transformers import AutoModel
TRAIN_FROM = 9
_probe = AutoModel.from_pretrained('microsoft/rad-dino')
N_BLOCKS = len(_probe.encoder.layer); D = _probe.config.hidden_size; del _probe
assert D == 768 and TRAIN_FROM < N_BLOCKS
N_PATCH = (MODEL_SIZE // 14) ** 2

assert os.path.exists(BB_CKPT), f'04b backbone khong thay: {BB_CKPT}'
_sd04b = torch.load(BB_CKPT, map_location='cpu', weights_only=False)['model']
BB_SD = {k[9:]: v for k, v in _sd04b.items() if k.startswith('backbone.')}
assert BB_SD, 'khong co backbone.* trong 04b checkpoint'

def load_backbone(device=dev):
    m = AutoModel.from_pretrained('microsoft/rad-dino')
    m.load_state_dict(BB_SD, strict=True)
    return m.to(device).eval()

backbone = load_backbone()
for p in backbone.parameters(): p.requires_grad_(False)
LAYERS = backbone.encoder.layer
def _layer(h, i):
    o = LAYERS[i](h); return o[0] if isinstance(o, (tuple, list)) else o

@torch.no_grad()
def forward_shared(x):
    h = backbone.embeddings(x)
    for i in range(TRAIN_FROM): h = _layer(h, i)
    return h.detach()

import sys, types
ML_SRC = '/content/ml_decoder.py'
if not os.path.exists(ML_SRC):
    !wget -q -O {ML_SRC} https://raw.githubusercontent.com/Alibaba-MIIL/ML_Decoder/main/src_files/ml_decoder/ml_decoder.py
sys.path.insert(0, '/content')
import ml_decoder as _mld
from ml_decoder import MLDecoder
if not hasattr(_mld.TransformerDecoderLayerOptimal, '_patched'):
    _oi = _mld.TransformerDecoderLayerOptimal.__init__
    def _init(self, *a, **kw): _oi(self, *a, **kw); self.self_attn = types.SimpleNamespace(batch_first=False)
    _mld.TransformerDecoderLayerOptimal.__init__ = _init
    _of = _mld.TransformerDecoderLayerOptimal.forward
    def _fwd(self, tgt, memory, **kw):
        kw.pop('tgt_is_causal', None); kw.pop('memory_is_causal', None); return _of(self, tgt, memory, **kw)
    _mld.TransformerDecoderLayerOptimal.forward = _fwd
    _mld.TransformerDecoderLayerOptimal._patched = True

assert os.path.exists(H5_CKPT), f'H5 head khong thay: {H5_CKPT}'
H5_SD = torch.load(H5_CKPT, map_location='cpu', weights_only=False)['model']
_sub = lambda p: {k[len(p):]: v for k, v in H5_SD.items() if k.startswith(p)}
assert _sub('norm.') and _sub('mld.')

_TAIL_SRC = load_backbone('cpu')
for p in _TAIL_SRC.parameters(): p.requires_grad_(False)

class TailBackbone(nn.Module):
    def __init__(s):
        super().__init__()
        s.blocks = nn.ModuleList([copy.deepcopy(_TAIL_SRC.encoder.layer[i])
                                  for i in range(TRAIN_FROM, N_BLOCKS)])
        s.ln = copy.deepcopy(_TAIL_SRC.layernorm)
        for p in s.parameters(): p.requires_grad_(True)
    def forward(s, h):
        for b in s.blocks:
            o = b(h); h = o[0] if isinstance(o, (tuple, list)) else o
        return s.ln(h)

class H5Head(nn.Module):
    def __init__(s):
        super().__init__()
        s.norm = nn.LayerNorm(D)
        s.mld = MLDecoder(num_classes=C, initial_num_features=D, decoder_embedding=D, num_of_groups=C)
        s.norm.load_state_dict(_sub('norm.')); s.mld.load_state_dict(_sub('mld.'))
    def forward(s, patches): return s.mld(s.norm(patches))

class Arm(nn.Module):
    def __init__(s):
        super().__init__(); s.tail = TailBackbone(); s.head = H5Head()
    def forward(s, z):
        h = s.tail(z); assert h.shape[1] == N_PATCH + 1
        return s.head(h[:, 1:])                       # PATCH-ONLY, bo CLS

def build_arm(seed=TEACHER_SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    return Arm().to(dev)

tail_params = lambda m: list(m.tail.parameters())
head_params = lambda m: list(m.head.parameters())

class CellWASL(nn.Module):
    '''ASL + weight tung o. w = mask -> dung cong thuc teacher §5 / hard-part cua student §10.'''
    def __init__(s): super().__init__(); s.gn, s.gp, s.clip, s.eps = ASL_GN, ASL_GP, ASL_CLIP, ASL_EPS
    def forward(s, x, y, w):
        y = y.float(); xp = torch.sigmoid(x); xn = (1-xp+s.clip).clamp(max=1)
        lp = y*torch.log(xp.clamp(min=s.eps)); ln = (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y); mod = torch.pow(1-pt, g)
        return (w*(-lp*mod - ln*mod)).sum()/(x.shape[0]*x.shape[1])       # mau so B_actual * 13
CRIT = CellWASL().to(dev)

def make_opt(m):
    def sp(ps, lr):
        d = [p for p in ps if p.ndim > 1]; nd = [p for p in ps if p.ndim <= 1]
        return [{'params': d, 'lr': lr, 'weight_decay': WD},
                {'params': nd, 'lr': lr, 'weight_decay': 0.0}]
    return torch.optim.AdamW(sp(tail_params(m), LR_BLOCKS) + sp(head_params(m), LR_HEAD))

def make_sched(opt, total, warm):
    cw = lambda s: s/max(warm, 1) if s < warm else 0.5*(1+math.cos(math.pi*(s-warm)/max(total-warm, 1)))
    return torch.optim.lr_scheduler.LambdaLR(opt, [cw, cw, lambda s: 1.0, lambda s: 1.0])

set_head_trainable = lambda m, on: [p.requires_grad_(on) for p in head_params(m)]
def clip_arm(m):
    t = [p for p in tail_params(m) if p.grad is not None]
    h = [p for p in head_params(m) if p.grad is not None]
    return (float(nn.utils.clip_grad_norm_(t, CLIP_BLOCKS)) if t else 0.0,
            float(nn.utils.clip_grad_norm_(h, CLIP_HEAD)) if h else 0.0)

# ---- G0: forward_shared + tail == backbone(x) ----
_A = build_arm(); _x = torch.randn(2, 3, MODEL_SIZE, MODEL_SIZE, device=dev)
with torch.no_grad():
    _d = (backbone(pixel_values=_x).last_hidden_state - _A.tail(forward_shared(_x))).abs().max().item()
print(f'G0 identity max|diff| {_d:.2e} {"OK" if _d < 1e-4 else "FAIL"}'); assert _d < 1e-4
_np_ = lambda ps: sum(p.numel() for p in ps)
print(f'trainable/teacher {_np_(tail_params(_A))+_np_(head_params(_A)):,}')
del _A; torch.cuda.empty_cache() if dev == 'cuda' else None

print(f'BATCH {BATCH} x ACCUM {ACCUM} = effective {BATCH*ACCUM} (muc tieu {EFF})')

## C7 · **Stage 1** — 5 teacher fold + OOF probability *(plan §6, §7)*

Fold `k`: train trên 4 fold còn lại (`TEACHER_EPOCHS` epoch, lịch R0, **không** chọn checkpoint bằng
fold held-out → không rò rỉ), rồi predict fold `k`, pool `0.7·F + 0.3·L` về study.

Resume theo epoch: ngắt Colab giữa chừng chạy lại cell là tiếp tục. Mỗi fold xong ghi ngay
`oof_fold{k}.npz`, nên có thể chạy rải nhiều session.

In [ ]:
Y_T_t = torch.from_numpy(Y_T).to(dev); M_T_t = torch.from_numpy(M_T).to(dev)

def pool_study_rows(p_img, rows):
    '''Pool anh -> study bang dung view pooling khoa cua R0 (W_F/W_L).'''
    sub = man.iloc[rows]
    d = pd.DataFrame(p_img, columns=LABELS)
    d['study_id'] = sub.study_id.to_numpy(); d['vg'] = sub.view_group.to_numpy()
    sid = np.sort(d.study_id.unique())
    fr = d[d.vg == 'frontal'].groupby('study_id')[LABELS].mean().reindex(sid)
    la = d[d.vg == 'lateral'].groupby('study_id')[LABELS].mean().reindex(sid)
    hf = fr.notna().all(1).to_numpy().astype('float32'); hl = la.notna().all(1).to_numpy().astype('float32')
    den = (W_F*hf + W_L*hl)[:, None]
    return sid, (W_F*np.nan_to_num(fr.to_numpy('float32'))*hf[:, None]
                 + W_L*np.nan_to_num(la.to_numpy('float32'))*hl[:, None]) / den

@torch.no_grad()
def predict_rows(m, rows, bs=None):
    m.eval(); bs = bs or BATCH
    dl = DataLoader(ImgDS(rows, eval_tf, Y_T, M_T), batch_size=bs, shuffle=False,
                    collate_fn=collate, num_workers=4, pin_memory=(dev == 'cuda'))
    P = np.zeros((len(rows), C), 'float32'); pos = 0
    for b in tqdm(dl, desc='oof pred', leave=False):
        b = to_dev(b); k = b['image'].shape[0]
        P[pos:pos+k] = torch.sigmoid(m(forward_shared(b['image']))).float().cpu().numpy(); pos += k
    assert pos == len(rows)
    return P

def train_teacher(fold):
    for _d in (f'{TEA_DIR}/checkpoints', f'{TEA_DIR}/preds'):
        os.makedirs(_d, exist_ok=True)       # KHONG dua vao C0: fold chay hang gio, Drive co the remount
    oof_p = f'{TEA_DIR}/preds/oof_fold{fold}.npz'
    if os.path.exists(oof_p):
        print(f'fold {fold}: <- cache {os.path.basename(oof_p)}'); return np.load(oof_p, allow_pickle=False)
    tr_rows = IDX['train'][man.fold.to_numpy()[IDX['train']] != fold]
    ho_rows = IDX['train'][man.fold.to_numpy()[IDX['train']] == fold]
    assert len(tr_rows) and len(ho_rows)
    _s_tr = set(man.study_id.to_numpy()[tr_rows]); _s_ho = set(man.study_id.to_numpy()[ho_rows])
    assert _s_tr.isdisjoint(_s_ho), 'study ro ri giua train/held-out cua teacher'
    assert set(man.subject_id.to_numpy()[tr_rows]).isdisjoint(set(man.subject_id.to_numpy()[ho_rows]))

    dl = loader(tr_rows, train_tf, True, BATCH, Y_T, M_T)
    STEPS = math.ceil(len(dl)/ACCUM); TOTAL = TEACHER_EPOCHS*STEPS; WARM = max(1, int(WARMUP_FRAC*TOTAL))
    rp = f'{TEA_DIR}/checkpoints/_resume_fold{fold}.pt'
    m = build_arm(TEACHER_SEED); opt = make_opt(m); sch = make_sched(opt, TOTAL, WARM); start = 0
    if os.path.exists(rp):
        ck = torch.load(rp, map_location=dev, weights_only=False)
        assert ck['batch'] == BATCH, f'resume batch {ck["batch"]} != {BATCH}; xoa {rp}'
        m.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch'])
        start = ck['epoch'] + 1; print(f'fold {fold}: RESUME @ep{start+1}')
    FROZEN0 = {k: v.detach().clone() for k, v in backbone.state_dict().items()}
    print(f'fold {fold}: train {len(tr_rows):,} anh / {len(_s_tr):,} study | '
          f'held-out {len(ho_rows):,} anh / {len(_s_ho):,} study | {STEPS:,} step/epoch')
    for ep in range(start, TEACHER_EPOCHS):
        unf = ep >= HEAD_WARMUP_EPOCHS
        set_head_trainable(m, unf); m.train()
        torch.manual_seed(TEACHER_SEED*7919 + fold*131 + ep)
        acc, nb, t0 = 0.0, 0, time.time()
        for i, b in enumerate(tqdm(dl, desc=f'f{fold} ep{ep+1}/{TEACHER_EPOCHS}', leave=False)):
            b = to_dev(b); z = forward_shared(b['image'])
            loss = CRIT(m(z), b['y'], b['m'])
            assert torch.isfinite(loss), f'HARD FAILURE: loss non-finite fold{fold} ep{ep+1} step{i}'
            (loss/ACCUM).backward(); acc += float(loss.detach()); nb += 1
            if ((i+1) % ACCUM == 0) or (i == len(dl)-1):
                nt, nh = clip_arm(m)
                assert math.isfinite(nt) and math.isfinite(nh), 'HARD FAILURE: grad norm'
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        assert not [k for k, v in backbone.state_dict().items() if not torch.equal(FROZEN0[k], v)], \
            'HARD FAILURE: blocks 0-8 bi doi'
        torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), sch=sch.state_dict(),
                        epoch=ep, batch=BATCH), rp+'.tmp'); os.replace(rp+'.tmp', rp)
        print(f'  fold{fold} ep{ep+1} loss {acc/max(nb,1):.5f} | {(time.time()-t0)/3600:.2f} h')
    torch.save({k: v.detach().cpu() for k, v in m.state_dict().items()},
               f'{TEA_DIR}/checkpoints/teacher_fold{fold}.pt')
    P = predict_rows(m, ho_rows)
    sid, PS = pool_study_rows(P, ho_rows)
    assert set(sid) == _s_ho, 'OOF thieu/thua study o fold held-out'
    np.savez(oof_p, sid=sid, p=PS.astype('float32'))
    del m, opt, sch; torch.cuda.empty_cache() if dev == 'cuda' else None
    print(f'fold {fold}: OOF {PS.shape} -> {os.path.basename(oof_p)}')
    return np.load(oof_p, allow_pickle=False)

RUN_TEACHER = True            # <<< dat False neu chi muon doc lai cache
_cached = sum(os.path.exists(f'{TEA_DIR}/preds/oof_fold{k}.npz') for k in range(K_FOLD))
print(f'teacher {TEACHER_TAG}: {_cached}/{K_FOLD} fold da co cache '
      f'-> {K_FOLD-_cached} fold can train')
OOF = {}
for k in range(K_FOLD):
    if RUN_TEACHER or os.path.exists(f'{TEA_DIR}/preds/oof_fold{k}.npz'):
        OOF[k] = train_teacher(k)

In [ ]:
# ---------------- gom OOF · assert plan §7 ----------------
sid_all = np.concatenate([OOF[k]['sid'] for k in range(K_FOLD)])
p_all   = np.concatenate([OOF[k]['p'] for k in range(K_FOLD)], 0)
fold_all = np.concatenate([np.full(len(OOF[k]['sid']), k, 'int8') for k in range(K_FOLD)])
assert len(sid_all) == len(np.unique(sid_all)), 'study bi lap trong OOF'
assert set(sid_all) == set(SID_TR.tolist()), 'OOF khong phu het TRAIN study'
_o = np.argsort(sid_all, kind='stable')
sid_all, p_all, fold_all = sid_all[_o], p_all[_o], fold_all[_o]
assert (sid_all == SID_TR).all()
assert np.isfinite(p_all).all(), 'OOF co NaN/Inf'

P_OOF = np.clip(p_all, P_CLIP, 1-P_CLIP).astype('float32')     # plan §7
assert P_OOF.shape == (N_TR, C) and (P_OOF > 0).all() and (P_OOF < 1).all()
assert (fold_all == FOLD_TR).all(), 'fold cua OOF lech fold map -> co study duoc teacher da train'

os.makedirs(TEA_DIR, exist_ok=True)
np.save(f'{TEA_DIR}/teacher_oof_train_probs.npy', P_OOF)
pd.DataFrame(dict(study_id=SID_TR, subject_id=SUBJ_TR, study_row=TR_ROWS, teacher_fold=FOLD_TR)
             ).to_parquet(f'{TEA_DIR}/teacher_oof_index.parquet', index=False)
print(f'OOF OK: {P_OOF.shape} | mot prediction/study | 0<p<1 | finite')
print(pd.DataFrame(dict(label=LABELS, mean=P_OOF.mean(0), p50=np.percentile(P_OOF, 50, 0),
                        p90=np.percentile(P_OOF, 90, 0), frac_gt_05=(P_OOF > .5).mean(0)))
      .to_string(index=False, float_format=lambda v: f'{v:.4f}'))

## C8 · **Stage 2** — final label manifest *(plan §8)*

`resolve_target(raw, label, blank_rate, teacher_p)` vector hoá theo từng nhãn. Xuất:

* `final_train_label_manifest.parquet` — long format, mỗi cell một dòng
* `Y_HARD/M_HARD/Y_SOFT/M_SOFT.npy` — `[N_cohort, 13]`, index bằng `study_row` để cắm thẳng vào loader

In [ ]:
P_OOF = np.load(f'{TEA_DIR}/teacher_oof_train_probs.npy')
RSN = ['explicit_positive', 'explicit_negative', 'uncertain_hard_positive', 'uncertain_teacher',
       'blank_hard_negative', 'blank_teacher', 'no_finding_positive', 'no_finding_negative',
       'no_finding_uncertain_mask', 'blank_hard_positive']
RSN_ID = {r: i for i, r in enumerate(RSN)}
TT_NAME = ['hard', 'soft', 'mask']

ttype  = np.full((N_TR, C), -1, 'int8')          # 0 hard | 1 soft | 2 mask
reason = np.full((N_TR, C), -1, 'int8')
final  = np.full((N_TR, C), np.nan, 'float32')

for j, l in enumerate(LABELS):
    st = ST_TR[:, j]; p = P_OOF[:, j]
    pos, neg, unc, bl = st == 1, st == 0, st == -1, st == 2
    is_nf = (l == NO_FINDING)
    # +1 / 0 — theo dung thu tu pseudocode §8: hai nhanh nay chan TRUOC nhanh No_Finding,
    # nen No_Finding raw +1/0 van mang reason explicit_* ('no_finding_positive' khong reachable).
    ttype[pos, j] = 0; final[pos, j] = 1.0; reason[pos, j] = RSN_ID['explicit_positive']
    ttype[neg, j] = 0; final[neg, j] = 0.0; reason[neg, j] = RSN_ID['explicit_negative']
    # -1
    if is_nf:
        ttype[unc, j] = 2; reason[unc, j] = RSN_ID['no_finding_uncertain_mask']
    elif l in HARD_U_ONES:
        ttype[unc, j] = 0; final[unc, j] = 1.0; reason[unc, j] = RSN_ID['uncertain_hard_positive']
    else:
        ttype[unc, j] = 1; final[unc, j] = p[unc]; reason[unc, j] = RSN_ID['uncertain_teacher']
    # blank
    if is_nf or policy[j] == 'HARD_ZERO':
        ttype[bl, j] = 0; final[bl, j] = 0.0
        reason[bl, j] = RSN_ID['no_finding_negative' if is_nf else 'blank_hard_negative']
    elif policy[j] == 'HARD_ONE':
        ttype[bl, j] = 0; final[bl, j] = 1.0; reason[bl, j] = RSN_ID['blank_hard_positive']
    else:
        ttype[bl, j] = 1; final[bl, j] = p[bl]; reason[bl, j] = RSN_ID['blank_teacher']

assert (ttype >= 0).all() and (reason >= 0).all()
assert np.isfinite(final[ttype != 2]).all() and np.isnan(final[ttype == 2]).all()

# ---------------- ma tran cho training (index bang study_row toan cohort) ----------------
Y_HARD = np.zeros((N, C), 'float32'); M_HARD = np.zeros((N, C), 'float32')
Y_SOFT = np.zeros((N, C), 'float32'); M_SOFT = np.zeros((N, C), 'float32')
Y_HARD[TR_ROWS] = np.where(ttype == 0, np.nan_to_num(final), 0.0)
M_HARD[TR_ROWS] = (ttype == 0).astype('float32')
Y_SOFT[TR_ROWS] = np.where(ttype == 1, np.nan_to_num(final), 0.0)
M_SOFT[TR_ROWS] = (ttype == 1).astype('float32')
assert not (M_HARD*M_SOFT).any(), 'mot cell vua hard vua soft'
os.makedirs(f'{LP1}/metrics', exist_ok=True)
for nm, a in [('Y_HARD', Y_HARD), ('M_HARD', M_HARD), ('Y_SOFT', Y_SOFT), ('M_SOFT', M_SOFT)]:
    np.save(f'{LP1}/{nm}.npy', a)

# ---------------- manifest long ----------------
_rep = lambda a: np.repeat(a, C)
MANI = pd.DataFrame(dict(
    subject_id=_rep(SUBJ_TR), study_id=_rep(SID_TR), study_row=_rep(TR_ROWS),
    label=pd.Categorical(np.tile(LABELS, N_TR), categories=LABELS),
    raw_label=ST_TR.reshape(-1), blank_rate=np.tile(blank_rate.astype('float32'), N_TR),
    neg_share=np.tile(neg_share.astype('float32'), N_TR),
    blank_policy=pd.Categorical(np.tile(policy, N_TR), categories=['HARD_ZERO', 'TEACHER_SOFT']),
    target_type=pd.Categorical(np.take(TT_NAME, ttype.reshape(-1)), categories=TT_NAME),
    final_target=final.reshape(-1), teacher_probability=P_OOF.reshape(-1),
    target_reason=pd.Categorical(np.take(RSN, reason.reshape(-1)), categories=RSN),
    teacher_fold=_rep(FOLD_TR.astype('int8'))))
MANI.to_parquet(f'{LP1}/final_train_label_manifest.parquet', index=False)
print(f'manifest {len(MANI):,} cell -> final_train_label_manifest.parquet')

SUMM = (MANI.groupby(['label', 'target_type'], observed=True).size().unstack(fill_value=0)
        .reindex(LABELS))
SUMM['soft_frac'] = SUMM.get('soft', 0) / SUMM.sum(1)
SUMM.to_csv(f'{LP1}/final_label_summary.csv')
print('\n=== FINAL LABEL SUMMARY (so cell theo target_type) ===')
print(SUMM.to_string(float_format=lambda v: f'{v:.4f}'))
print('\n=== theo target_reason ===')
print(MANI.target_reason.value_counts().to_string())

## C9 · Diagnostics bắt buộc *(plan §15)* — soft-target · effective prevalence · leakage

In [ ]:
# ---------------- soft-target summary, tach uncertain_teacher / blank_teacher ----------------
rows = []
for j, l in enumerate(LABELS):
    for rn in ('uncertain_teacher', 'blank_teacher'):
        m = reason[:, j] == RSN_ID[rn]
        if not m.any():
            rows.append(dict(label=l, source=rn, count=0)); continue
        v = final[m, j]
        rows.append(dict(label=l, source=rn, count=int(m.sum()), mean=float(v.mean()),
                         std=float(v.std()),
                         **{f'p{q:02d}': float(np.percentile(v, q)) for q in (1, 10, 25, 50, 75, 90, 99)},
                         frac_gt_05=float((v > .5).mean()), frac_gt_09=float((v > .9).mean())))
SOFT = pd.DataFrame(rows); SOFT.to_csv(f'{LP1}/teacher_soft_target_summary.csv', index=False)
print('=== SOFT TARGET SUMMARY ===')
print(SOFT.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

# ---------------- effective prevalence ----------------
hist_prev  = ((ST_TR == 1) | (ST_TR == -1)).mean(0)                       # U-Ones historical
thard_prev = np.array([Y_T[TR_ROWS][M_T[TR_ROWS][:, j] == 1, j].mean() for j in range(C)])
eff_prev   = (np.where(ttype == 0, np.nan_to_num(final), 0.0).sum(0)
              + np.where(ttype == 1, np.nan_to_num(final), 0.0).sum(0)) / N_TR
EFF_P = pd.DataFrame(dict(label=LABELS, historical=hist_prev, teacher_hard=thard_prev,
                          effective=eff_prev, ratio=eff_prev/np.maximum(hist_prev, 1e-9)))
EFF_P['flag'] = np.where(EFF_P.ratio > 2, 'HIGH >2x', np.where(EFF_P.ratio < 0.5, 'LOW <0.5x', ''))
EFF_P.to_csv(f'{LP1}/effective_prevalence.csv', index=False)
print('\n=== EFFECTIVE PREVALENCE (plan §15) ===')
print(EFF_P.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
_w = EFF_P[EFF_P.flag != '']
if len(_w):
    print(f'\n!! CANH BAO {len(_w)}/{C} nhan lech prevalence: {list(_w.label)}')
    print('   Plan §15: canh bao KHONG tu dong doi policy. Ghi nhan va bao cao.')

# ---------------- leakage checks ----------------
print('\n=== LEAKAGE CHECKS (plan §15) ===')
chk = {}
chk['subject disjoint across folds'] = all(
    set(SUBJ_TR[FOLD_TR == k]).isdisjoint(set(SUBJ_TR[FOLD_TR != k])) for k in range(K_FOLD))
_tr_sid = set(SID_TR.tolist())
chk['VAL/TEST khong nam trong teacher train'] = _tr_sid.isdisjoint(
    set(ehr.study_id.to_numpy()[~IS_TRAIN].tolist()))
chk['blank rate chi tu TRAIN'] = bool(np.allclose(blank_rate, (STATE[IS_TRAIN] == 2).mean(0)))
chk['moi TRAIN study co dung 1 OOF'] = (len(sid_all) == N_TR and (sid_all == SID_TR).all())
chk['OOF sinh boi teacher chua train tren study do'] = bool((fold_all == FOLD_TR).all())
chk['mask/hard/soft khong chong lap'] = bool(not (M_HARD*M_SOFT).any())
for k, v in chk.items(): print(f'  {"OK  " if v else "FAIL"} {k}')
assert all(chk.values()), 'leakage check FAIL'
json.dump({k: bool(v) for k, v in chk.items()}, open(f'{LP1}/metrics/leakage_checks.json', 'w'), indent=1)

## C10 · **Stage 3** — student loss + `experiment_manifest.json`

\[ L=\frac{\sum M^{hard}L_{ASL}(Y^{hard},z)+\lambda_{soft}\sum M^{soft}L_{BCE}(Y^{soft},z)}{B_{actual}\times 13} \]

Masked cell đóng góp đúng 0. Không renormalize theo số cell. Teacher prob là hằng số.

Cắm vào vòng train R0 của nb16: đổi `ImgDS` sang trả `(yh, mh, ys, ms)` theo `study_row`,
đổi `CRIT(zz, b['y'], Wt[a][b['sr']])` thành `student_loss(zz, ...)`. Mọi thứ khác giữ nguyên.

In [ ]:
# C10 phai chay duoc CA KHI bo qua C5/C6 (luot chi sinh nhan, teacher da cache).
if 'CRIT' not in dir():
    class CellWASL(nn.Module):
        def __init__(s): super().__init__(); s.gn, s.gp, s.clip, s.eps = ASL_GN, ASL_GP, ASL_CLIP, ASL_EPS
        def forward(s, x, y, w):
            y = y.float(); xp = torch.sigmoid(x); xn = (1-xp+s.clip).clamp(max=1)
            lp = y*torch.log(xp.clamp(min=s.eps)); ln = (1-y)*torch.log(xn.clamp(min=s.eps))
            pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y); mod = torch.pow(1-pt, g)
            return (w*(-lp*mod - ln*mod)).sum()/(x.shape[0]*x.shape[1])
    CRIT = CellWASL().to(dev)
    print('CRIT dinh nghia tai cho (bo qua C6)')

YH_t = torch.from_numpy(Y_HARD).to(dev); MH_t = torch.from_numpy(M_HARD).to(dev)
YS_t = torch.from_numpy(Y_SOFT).to(dev); MS_t = torch.from_numpy(M_SOFT).to(dev)

def student_loss(z, sr):
    '''H1 loss. z [B,13] logits; sr [B] study_row.'''
    yh, mh, ys, ms = YH_t[sr], MH_t[sr], YS_t[sr], MS_t[sr]
    hard = CRIT(z, yh, mh)                                    # da chia B*13
    soft = (ms*F.binary_cross_entropy_with_logits(z, ys, reduction='none')).sum()/(z.shape[0]*z.shape[1])
    return hard + LAMBDA_SOFT*soft

# ---- kiem tra: masked cell dong gop dung 0 ----
_sr = torch.from_numpy(TR_ROWS[:8]).to(dev)
_z = torch.randn(8, C, device=dev, requires_grad=True)
_l = student_loss(_z, _sr); _l.backward()
_g = _z.grad.detach().cpu().numpy()
_mask = (M_HARD[TR_ROWS[:8]] + M_SOFT[TR_ROWS[:8]]) == 0
print(f'masked cell: {int(_mask.sum())} | max|grad| tren masked = {np.abs(_g[_mask]).max() if _mask.any() else 0:.2e}')
assert not _mask.any() or np.abs(_g[_mask]).max() == 0.0, 'masked cell van co gradient'
print(f'student_loss OK | lambda_soft {LAMBDA_SOFT} | mau so B*{C}')

# ---------------- experiment manifest (plan §16) ----------------
def sha_file(p, buf=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while True:
            b = f.read(buf)
            if not b: break
            h.update(b)
    return h.hexdigest()
sha_obj = lambda o: hashlib.sha256(json.dumps(o, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

CODE_COMMIT = None
try:
    import subprocess
    CODE_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True,
                                 timeout=10).stdout.strip() or None
except Exception:
    pass

_ck = {f'teacher_fold{k}.pt': f'{TEA_DIR}/checkpoints/teacher_fold{k}.pt' for k in range(K_FOLD)}
EXP = dict(
    plan='LABEL_PIPELINE_V1_DETAILED_PLAN.md', review='REVIEW_LABEL_PIPELINE_V1.md',
    raw_label_file=RAW_PQ, raw_label_sha=sha_file(RAW_PQ),
    split_sha={s: sha_obj(np.sort(ehr.study_id.to_numpy()[SPLIT == s]).tolist())
               for s in ('train', 'val', 'test')},
    blank_route_rule=BLANK_ROUTE_RULE, neg_share_threshold=NEG_SHARE_THRESH,
    blank_threshold_plan_superseded=BLANK_THRESH_PLAN,
    blank_rate_and_policy={l: dict(blank_rate=float(r), neg_share=float(q), policy=str(p))
                           for l, r, q, p in zip(LABELS, blank_rate, neg_share, policy)},
    hard_u_ones=sorted(HARD_U_ONES), soft_u_labels=sorted(SOFT_U_LABELS),
    teacher_fold_map_sha=sha_obj(FOLD_TR.tolist()), teacher_seed=TEACHER_SEED,
    teacher_epochs=TEACHER_EPOCHS, teacher_ckpt_rule=TEACHER_CKPT_RULE,
    teacher_mode=TEACHER_MODE, teacher_tag=TEACHER_TAG, teacher_dir=TEA_DIR,
    teacher_ckpt_sha={n: sha_file(p) for n, p in _ck.items() if os.path.exists(p)},
    oof_sha=sha_file(f'{TEA_DIR}/teacher_oof_train_probs.npy'),
    manifest_sha=sha_file(f'{LP1}/final_train_label_manifest.parquet'),
    lambda_soft=LAMBDA_SOFT, student_seeds=STUDENT_SEEDS, code_commit=CODE_COMMIT,
    labels=LABELS, batch=BATCH, accum=ACCUM, effective_batch=BATCH*ACCUM,
    n_train_studies=int(N_TR),
    n_cells=dict(hard=int(M_HARD[TR_ROWS].sum()), soft=int(M_SOFT[TR_ROWS].sum()),
                 mask=int(((M_HARD+M_SOFT)[TR_ROWS] == 0).sum())))
json.dump(EXP, open(f'{LP1}/experiment_manifest.json', 'w'), indent=1)
print(json.dumps({k: v for k, v in EXP.items() if k not in ('blank_rate_and_policy', 'labels')},
                 indent=1)[:2000])
print(f'\n-> {LP1}/experiment_manifest.json')
print('\nStage 3: train H1 bang setup R0 (nb16 C12) voi student_loss, 3 seed 42/1337/2024,')
print('         eval historical VAL. TEST chi mo sau khi qua cong §14.')